In [0]:
%sql
-- This will drop the database and all its tables
DROP DATABASE IF EXISTS ehr_union CASCADE;

-- Recreate the empty database
CREATE DATABASE ehr_union;

In [0]:
import sys
sys.path.append("/Workspace/Shared/ai-readi-ehr-omop-pipeline/logic/")
from myproject import final_schema
from pyspark.sql.functions import col, count
from pyspark.sql import functions as F, types as T
from myproject import timestamp_comment

def perform_union(spark, domain, **inputs):
    """Union multiple dataframes with proper schema enforcement"""
    # Get schema from final_schema module
    schema = final_schema.schema_dict[domain]
    
    # Enforce schema for each input dataframe
    dfs_to_union = []
    for site_name, df_site in inputs.items():
        try:
            # Select only columns that exist in the schema
            df_with_schema = df_site.selectExpr(*schema)
            dfs_to_union.append(df_with_schema)
            print(f"Successfully processed {site_name} for domain {domain}")
        except Exception as e:
            print(f"Error processing {site_name} for domain {domain}: {str(e)}")
            # Continue with other dataframes instead of failing completely
            continue
    
    # Check if we have any dataframes to union
    if not dfs_to_union:
        raise ValueError(f"No valid input dataframes found for domain {domain}")
    
    # Union all dataframes together
    unioned_df = dfs_to_union[0]
    for i, df in enumerate(dfs_to_union[1:], 1):
        try:
            unioned_df = unioned_df.unionByName(df, allowMissingColumns=True)
            
        except Exception as e:
            print(f"Error during union operation {i+1}: {str(e)}")
            raise
    
    # Ensure final column ordering matches schema
    unioned_df = unioned_df.selectExpr(*schema)
    
    return unioned_df


def get_inputs_databricks(spark, domain):
    inputs = {}
    #inputs["jhu"] = spark.table(f"jhu_staging.{domain}")
    inputs["uab"] = spark.table(f"uab_staging.{domain}")
    inputs["uw"] = spark.table(f"uw_staging.{domain}")
    inputs["ucsd"] = spark.table(f"ucsd_staging.{domain}")
    #print(f"union sites files: {list(inputs.keys())}")
    return inputs


def validate_primary_key(df, key_col):
    """Check if the primary key contains unique values"""
    count_all = df.count()
    count_distinct = df.select(key_col).distinct().count()
    if count_all != count_distinct:
        raise ValueError(f"Column {key_col} is not a valid primary key")
    else:
        print(f"Column {key_col} is a valid primary key")
    return True


def my_compute_function(domain, key_col):
    """Transform function to union datasets together"""
    inputs = get_inputs_databricks(spark, domain)  # Added spark parameter
    
    df_out = perform_union(spark, domain, **inputs)  # Added spark parameter
    
    validate_primary_key(df_out, key_col)
    
    output_table = f"EHR_union.{domain}"
    df_out.write.mode("overwrite").saveAsTable(output_table)

    # Add comment separately
    timestamp_comment.comment(spark,"EHR_union",domain)
    return df_out


domains = [
    # domain, pkey, 
    ("care_site", "care_site_id"),
    # ("condition_era", "condition_era_id"),
    ("condition_occurrence", "condition_occurrence_id"),
    # ("control_map", 'control_map_id'),
    ("death", 'person_id'),
    ("device_exposure", "device_exposure_id"),
    # ("dose_era", "dose_era_id"),
    # ("drug_era", "drug_era_id"),
    ("drug_exposure", "drug_exposure_id"),
    ("location", "location_id"),
    ("measurement", "measurement_id"),
    # ("note", 'note_id'),  # Temporarily do not check the primary key - note_id
    # ("note_nlp",'note_nlp_id'),  # Temporarily do not check the primary key - note_nlp_id
    ("observation", "observation_id"),
    # ("observation_period", "observation_period_id"),
    ("person", "person_id"),
    ("procedure_occurrence", "procedure_occurrence_id"),
    ("provider", "provider_id"),
    # ("visit_detail", "visit_detail_id"),
    ("visit_occurrence", "visit_occurrence_id"),
]

for domain_tuple in domains:
    domain_name = domain_tuple[0]
    key_col = domain_tuple[1]
    my_compute_function(domain_name, key_col)
    print(f"Processed {domain_name}...")

In [0]:
# union files for DQ dashboard
df= spark.sql("""select * from uab_ingestion.07_pre_clean_removed_person_ids
              union all
              select * from uw_ingestion.07_pre_clean_removed_person_ids
              union all
              select * from ucsd_ingestion.07_pre_clean_removed_person_ids""")
df.write.format("delta").mode("overwrite").saveAsTable("ehr_dqp.person_id_remove")
timestamp_comment.comment(spark,'ehr_dqp','person_id_remove')